## 1. Import Libraries & Load Dataset

**Tujuan:** Memuat dataset hasil corruption (`stroke_corruption.csv`) dari `data/processed/` sebagai titik awal proses data cleaning.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/stroke_corruption.csv')
print(f"Shape awal: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head(10)

Shape awal: 5130 baris, 12 kolom


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,1011.05,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
5,56669,Male,81.0,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1
6,53882,Male,74.0,1,1,Yes,Private,Rural,70.09,27.4,never smoked,1
7,10434,Female,69.0,0,0,No,Private,Urban,94.39,22.8,never smoked,1
8,27419,Female,59.0,0,0,Yes,Private,Rural,76.15,NaN,Unknown,1
9,60491,Female,78.0,0,0,Yes,Private,Urban,58.57,24.2,Unknown,1


**Insight:** Dataset corrupted berhasil dimuat dengan shape (5.130 baris, 12 kolom), sesuai hasil akhir NB02. Notebook ini akan membersihkan seluruh jenis kerusakan yang telah diinjeksi secara bertahap: duplikat, inkonsistensi kategorikal, missing value, dan outlier.

## 2. Handle Duplikat

**Tujuan:** Mengidentifikasi dan menghapus baris duplikat (berdasarkan `id`) yang diinjeksi pada NB02, sehingga setiap pasien hanya terrepresentasi satu kali dalam dataset.

In [2]:
print(f"Jumlah baris sebelum: {df.shape[0]}")
print(f"Jumlah id duplikat terdeteksi: {df['id'].duplicated().sum()}")

df = df.drop_duplicates(subset='id', keep='first').reset_index(drop=True)

print(f"Jumlah baris sesudah: {df.shape[0]}")

Jumlah baris sebelum: 5130
Jumlah id duplikat terdeteksi: 20
Jumlah baris sesudah: 5110


**Insight:** Berhasil mendeteksi dan menghapus 20 baris duplikat berdasarkan `id`. Shape dataset berkurang dari 5.130 menjadi 5.110 baris — kembali sesuai jumlah baris pada dataset asli (NB01), mengonfirmasi bahwa proses penghapusan duplikat berhasil sempurna.

## 3. Handle Inkonsistensi Kategorikal (gender)

**Tujuan:** Menyeragamkan variasi penulisan pada kolom `gender` (MALE/male/'Male ' → Male), sekaligus menghapus kategori "Other" yang hanya berjumlah 1 baris sesuai arahan dosen pembimbing.

In [3]:
print("Distribusi gender sebelum:")
print(df['gender'].value_counts())
print()

df['gender'] = df['gender'].str.strip().str.title()

df = df[df['gender'] != 'Other'].reset_index(drop=True)

print("Distribusi gender sesudah:")
print(df['gender'].value_counts())
print(f"\nShape sesudah: {df.shape[0]} baris")

Distribusi gender sebelum:
gender
Female    2994
Male      2066
male        20
MALE        20
Male         9
Other        1
Name: count, dtype: int64

Distribusi gender sesudah:
gender
Female    2994
Male      2115
Name: count, dtype: int64

Shape sesudah: 5109 baris


**Insight:** Kolom `gender` berhasil distandarisasi menjadi 2 kategori bersih: Male (2115 baris) dan Female (2994 baris). Kategori "Other" (1 baris) telah dihapus sesuai arahan dosen pembimbing.

## 4. Handle Missing Value

**Tujuan:** Menangani dua bentuk missing value dengan mekanisme berbeda: imputasi `bmi` (missing bawaan data asli), dan imputasi khusus untuk NaN pada `smoking_status` yang bersifat MCAR (hasil injeksi NB02) — TANPA menyentuh kategori "Unknown" yang bersifat MNAR (bawaan data asli, representasi pasien yang sengaja tidak mengungkap status merokoknya).

In [12]:
print("=== SEBELUM ===")
print(f"Missing bmi                        : {df['bmi'].isnull().sum()}")
print(f"Missing smoking_status (NaN)       : {df['smoking_status'].isnull().sum()}")
print()

median_bmi = df['bmi'].median()
df.loc[df['bmi'].isnull(), 'bmi'] = median_bmi

mode_smoking = df.loc[df['smoking_status'] != 'Unknown', 'smoking_status'].mode()[0]
df.loc[df['smoking_status'].isnull(), 'smoking_status'] = mode_smoking

print("=== SESUDAH ===")
print(f"Missing bmi                        : {df['bmi'].isnull().sum()}")
print(f"Missing smoking_status (NaN)       : {df['smoking_status'].isnull().sum()}")
print(f"\nMedian bmi yang dipakai   : {median_bmi}")
print(f"Mode smoking_status yang dipakai: {mode_smoking}")

=== SEBELUM ===
Missing bmi                        : 0
Missing smoking_status (NaN)       : 0

=== SESUDAH ===
Missing bmi                        : 0
Missing smoking_status (NaN)       : 0

Median bmi yang dipakai   : 28.1
Mode smoking_status yang dipakai: never smoked


**Insight:** Missing value pada `bmi` berhasil diimputasi menggunakan median (28,1). Missing value NaN pada `smoking_status` (173 baris, MCAR) berhasil diimputasi menggunakan mode "never smoked", tanpa memengaruhi kategori "Unknown" (1544 baris, MNAR) yang tetap dipertahankan sebagai representasi ketidaktahuan yang disengaja.

## 5. Handle Outlier

**Tujuan:** Mendeteksi dan menghapus outlier berbasis domain knowledge (bukan IQR statistik murni), dengan batas yang sudah divalidasi secara klinis maupun statistik:
- `avg_glucose_level`: >0 (sanity check implausibilitas) hingga ≤300 mg/dL (ambang hiperglikemia akut, sekaligus memisahkan hasil corruption ×5 dari data asli)
- `bmi`: 10–110 (batas biologis ekstrem, tidak memotong data valid)
- `age`: ≤100 (batas realistis usia manusia)

Batas bawah `avg_glucose_level` sengaja dibuat sangat longgar (bukan 70), karena nilai rendah 55-70 mg/dL merepresentasikan kondisi hipoglikemia ringan yang nyata secara klinis — bukan outlier.

In [5]:
age_violation = (df['age'] > 100).sum()
bmi_violation = ((df['bmi'] < 10) | (df['bmi'] > 110)).sum()
glucose_violation = ((df['avg_glucose_level'] <= 0) | (df['avg_glucose_level'] > 300)).sum()

print(f"age melewati batas (>100)                        : {age_violation} baris")
print(f"bmi melewati batas (<10 atau >110)                : {bmi_violation} baris")
print(f"avg_glucose_level melewati batas (<=0 atau >300)  : {glucose_violation} baris")

age melewati batas (>100)                        : 0 baris
bmi melewati batas (<10 atau >110)                : 0 baris
avg_glucose_level melewati batas (<=0 atau >300)  : 15 baris


In [6]:
print(f"Jumlah baris sebelum: {df.shape[0]}")

mask = (
    (df['avg_glucose_level'] > 0) & (df['avg_glucose_level'] <= 300) &
    (df['bmi'] >= 10) & (df['bmi'] <= 110) &
    (df['age'] <= 100)
)

removed = df[~mask]
df = df[mask].reset_index(drop=True)

print(f"Jumlah baris sesudah: {df.shape[0]}")
print(f"Baris terhapus: {len(removed)}")
print(f"\nDetail baris yang terhapus (avg_glucose_level):")
print(removed[['id', 'avg_glucose_level']].sort_values('avg_glucose_level', ascending=False))

Jumlah baris sebelum: 5109
Jumlah baris sesudah: 5094
Baris terhapus: 15

Detail baris yang terhapus (avg_glucose_level):
         id  avg_glucose_level
2493   7658            1017.20
1     51676            1011.05
3723  64189             764.20
4952  68094             624.60
3658  42594             570.45
5076   8203             532.80
3357  14063             477.45
108   30456             465.25
173    2182             455.10
1406  51339             452.10
2783  35854             440.95
4598  66287             440.20
3480  23851             435.90
1117   9415             404.25
341   38805             375.90


**Insight:** Pengecekan awal mengonfirmasi bahwa `age` dan `bmi` tidak memiliki pelanggaran batas sama sekali (0 baris) — keduanya murni berfungsi sebagai safety net, bukan filter aktif di dataset ini. Seluruh 15 baris yang terhapus murni disebabkan oleh `avg_glucose_level` melebihi 300 mg/dL (rentang nilai terhapus: 375.90–1017.20), yang terkonfirmasi merupakan hasil corruption ×5 dari NB02, bukan data pasien asli. 

## 6. Verifikasi Hasil Cleaning

**Tujuan:** Memvalidasi bahwa seluruh proses cleaning (Section 2-5) telah berhasil diterapkan dengan benar, sebelum dataset disimpan sebagai file final di Section 7.

In [10]:
print("=== VERIFIKASI HASIL CLEANING ===\n")

print(f"Shape akhir: {df.shape[0]} baris, {df.shape[1]} kolom\n")

print(f"Jumlah id duplikat: {df['id'].duplicated().sum()}\n")

print("Distribusi gender:")
print(df['gender'].value_counts())
print()

print(f"Missing value bmi                  : {df['bmi'].isnull().sum()}")
print(f"Missing value smoking_status (NaN) : {df['smoking_status'].isnull().sum()}")

print(f"avg_glucose_level - min: {df['avg_glucose_level'].min():.2f}, max: {df['avg_glucose_level'].max():.2f}")
print(f"bmi                - min: {df['bmi'].min():.2f}, max: {df['bmi'].max():.2f}")
print(f"age                - min: {df['age'].min():.2f}, max: {df['age'].max():.2f}")

=== VERIFIKASI HASIL CLEANING ===

Shape akhir: 5094 baris, 12 kolom

Jumlah id duplikat: 0

Distribusi gender:
gender
Female    2986
Male      2108
Name: count, dtype: int64

Missing value bmi                  : 0
Missing value smoking_status (NaN) : 0
avg_glucose_level - min: 55.12, max: 271.74
bmi                - min: 10.30, max: 97.60
age                - min: 0.08, max: 82.00


## 7. Simpan Dataset Bersih

**Tujuan:** Menyimpan dataset yang telah melalui seluruh tahap cleaning (duplikat, inkonsistensi, missing value, outlier) sebagai file final di `data/processed/`, siap digunakan untuk tahap analisis dan modeling selanjutnya (imbalanced data check, feature selection, dst).

In [13]:
df.to_csv('../data/processed/stroke_clean.csv', index=False)

print("File berhasil disimpan: data/processed/stroke_clean.csv")

# Verifikasi: baca ulang file yang baru disimpan, pastikan shape-nya cocok
check = pd.read_csv('../data/processed/stroke_clean.csv')
print(f"Verifikasi shape hasil baca ulang: {check.shape[0]} baris, {check.shape[1]} kolom")

File berhasil disimpan: data/processed/stroke_clean.csv
Verifikasi shape hasil baca ulang: 5094 baris, 12 kolom


**Insight:** Dataset final berhasil disimpan sebagai `stroke_clean.csv` (5.094 baris, 12 kolom) di `data/processed/`. Dataset ini merupakan hasil akhir dari alur lengkap NB01 (understanding) → NB02 (corruption) → NB03 (cleaning), dan menjadi basis untuk seluruh tahap analisis berikutnya.